## Before you start — switch this notebook to the R runtime

Google Colab defaults to a Python runtime. Because this problem set is written in R, you need to switch the runtime once before running any code:

1. In the Colab menu bar, click **Runtime → Change runtime type**.
2. Under **Runtime type**, select **R**.
3. Click **Save**. Colab will reload the notebook with R.

You only need to do this once per Colab session. After switching, run the next cell to install the R packages you'll need.


In [ ]:
# Fetch this assignment's data files from the template repo.
# Needed on Google Colab (which only loads the notebook itself);
# harmless when the data is already present (e.g. running locally).
# Pulls in: `data/`
system("if [ ! -d data ]; then rm -rf /tmp/_pset_data && git clone --depth 1 --quiet https://github.com/yse-eds-cert/yse-eds-cert-classroom-assignment-5-practice-set-template.git /tmp/_pset_data && cp -r /tmp/_pset_data/data ./ && rm -rf /tmp/_pset_data; fi")

## Data Stewardship -- Problem Set 5: Efficient Coding and Data Manipulation in R
In this problem set, you will clean the raw air quality measurement from the `example_air_quality.csv` dataset, join them with additional information about each monitoring station, and analyze difference in PM2.5 concentrations across the locations. Recall from lecture that the CSV contains the following variables:

    - timestamp: the date and time of each measurement
    - location: the monitoring site (North, South, East, or West Station)
    - temperature_C: temperature in degrees Celsius
    - humidity_percent: relative humidity (%)
    - pm25: fine particulate matter concentration (µg/m³)
    - no2_ppb: nitrogen dioxide (parts per billion)
    - ozone_ppb: ozone (parts per billion)


Throughout this problem set, be sure to include comments with meaningful descriptions for each step. 

As always, use sources like Google, Stack Overflow, documentation, and other forums for help. Try to avoid AI in order to develop your own ability to conceptualize solution strategies.

## Question 0: Set-up Working Directory (3 points)

Before starting work on your data proper, it is good practice to check what working directory (i.e. folder) your code thinks it is working in. The **working directory** is the folder R looks in by default when you ask it to read or write files. If R can't find your data, the working directory is usually the first thing to check.

We've filled in this question for you so you can see the standard set-up workflow. Read through each step and run the cells so you understand what's happening — you'll use these same tools throughout the problem set.

1. `getwd()` returns the **current working directory** as a string. We save it in `my_wd` so we can return to it later. (Run the cell to see the path.)

2. `list.files()` lists the files and folders contained in a given directory. We pass it `my_wd` and save the result in `my_files` to see what's in our working directory — notice that there's a `data` folder.

3. The data file `example_air_quality.csv` lives inside that `data` folder, not in the working directory itself. We use `setwd('data')` to step into that subfolder, then `list.files()` again (saved as `locate_data`) to confirm the CSV is there.

Finally, we navigate back to the original working directory with `setwd(my_wd)` so the rest of the notebook runs from a known location.

> **Why this matters:** in Question 1 you'll need to give `read.csv()` the correct path to the data. Knowing where you are (`getwd()`) and what's around you (`list.files()`) is how you figure out the right path — e.g. `"data/example_air_quality.csv"`.


In [0]:
# check current working directory
my_wd <- getwd()
my_wd

In [0]:
# check files in current working directory
my_files <- list.files(my_wd)
my_files

In [0]:
# locate example_air_quality.csv
setwd('data')
locate_data <- list.files(getwd())
locate_data

In [0]:
# Navigate back to your original working directory
setwd(my_wd)
# pro tip - you can also run: setwd("../") "../" means "one directory level up".

In [0]:
# your grade is based on the following tests: 

# Question 0 test 
if (!'example_air_quality.csv' %in% locate_data){
    stop("You didn't locate the data directory.")
} else{
    print("Nice work, your data is read in correctly.")
}

## Question 1: Code Organization (3 points)

Start your notebook by setting it up in a clean and organized way with clear, descriptive naming:

1. Load the relevant libraries for data cleaning and visualization.
2. Import `example_air_quality.csv`.  
   - If you are running into problems at this step - what is your current working directory? What directory is `example_air_quality.csv` in? How should you change the input to your `read.csv()` function to account for the difference?
3. Inspect the dataset to refresh your memory of its variables and data types.

Recall from your lecture videos this week: what was the main problem Mitch discussed with respect to the `example_air_quality.csv` dataset?

4. Create a new cleaned dataframe that:
   - Fixes this problem for the `pm25` or `timestamp` columns (it should take no longer than 2-3 lines of code), and
   - Converts the `timestamp` column from a character string into a proper date/datetime format using `mutate(timestamp = as.POSIXct(timestamp, format = "%m/%d/%y %H:%M"))`

Recall from Data Foundations last week that `mutate()` is the tidyverse function for creating a new column in your dataset.

Feel free to use additional sources to learn more about date and datetime formats in R: https://stat.ethz.ch/R-manual/R-devel/library/base/html/as.POSIXlt.html.


In [0]:
# load libraries
# YOUR CODE HERE
stop("No Answer Given!")

In [0]:
# import data
# uncomment the below line of code to get started
air_quality_df <- read.csv(YOUR DATA FILE PATH HERE)
# YOUR CODE HERE
stop("No Answer Given!")

In [0]:
# inspect the data
# YOUR CODE HERE
stop("No Answer Given!")

In [0]:
# clean data
# uncomment the below line of code to get started
# clean_AQ_df <- air_quality_df %>%
    # Fix problem for pm25 and timestamp columns
    # YOUR CODE HERE
    stop("No Answer Given!")
    # convert timestamp column 
    # YOUR CODE HERE
    stop("No Answer Given!")

In [0]:
# your grade is based on the following tests: 

# Question 1 test 
if (nrow(clean_AQ_df)!=93 | sum(is.na(clean_AQ_df$timestamp))+sum(is.na(clean_AQ_df$pm25))!=0){
    stop("You did not clean the dataframe correctly.")
} else if (class(clean_AQ_df$timestamp)[1]!='POSIXct'){
    stop("You did not convert the timestamp column correctly.")
} else{
    print("Nice work, you cleaned the dataframe correctly.")
}

## Question 2: Creating a Metadata Table for Joins (2 points)

Now, we have a cleaned dataset with PM2.5 measurements at the four monitoring stations, but those values alone don’t explain why air quality might differ from place to place. To better interpret the data, we need more information about the stations themselves.

First, we’ll assign each station a site type to describe its surrounding environment:

    - North Station → Urban
    - South Station → Near-road
    - West Station → Suburban
    - East Station → Waterfront


Second, air quality monitors can sometimes give measurements that are different from the true PM2.5 concentration in the air. Small differences in the type of sensor, how old it is, or how well it has been maintained can affect the numbers it reports.

To make the data comparable across locations, researchers often apply calibration factors. A calibration factor is a simple multiplier that adjusts each monitor’s readings, so they are more consistent with a trusted baseline.

For example, if a sensor tends to read 5% too high, a calibration factor of ~0.95 (1.00/1.05) will scale it down. If a sensor reads a bit too low, perhaps 97% of the true value, a factor near 1.03 (1.00/0.97) will scale it up. These adjustments make values across sites more directly comparable.

For our four stations, we’ll use the following sample calibration factors:

    - North Station → 0.98 (slightly high, scaled down 2%)
    - South Station → 1.03 (slightly low, scaled up 3%)
    - West Station → 1.00 (well calibrated, no adjustment)
    - East Station → 1.05 (reads low, scaled up 5%)

Now, create a dataframe called `site_meta` containing three columns with clear, descriptive names corresponding to

    - Location (station name - make sure you use the exact same text for both column name and column values as your cleaned data frame from Q1!)
    - Site Type (as listed above)
    - Calibration Factor (as listed above)

Make sure your dataframe has the rows and columns in the exact order above, otherwise you may not receive credit for your answer.

Lastly, print the `site_meta` dataframe.

In [0]:
# create a dataframe
# YOUR CODE HERE
stop("No Answer Given!")

# print the dataframe - just uncomment the line of code below
# site_meta


In [0]:
# your grade is based on the following tests: 

# Question 2 test 
if (!all(site_meta[1]==c("North Station", "South Station", "West Station", "East Station"))){
    stop("You did not create the correct first dataframe column.")
} else if (!all(site_meta[2]==c("Urban", "Near-road", "Suburban", "Waterfront"))){
    stop("You did not create the correct second dataframe column.")
} else if (!all(site_meta[3]==c(0.98, 1.03, 1.00, 1.05))){
    stop("You did not create the correct third dataframe column.")
} else {
    print("Nice work, you created the dataframe correctly.")
}


## Question 3: Joins (4 points)

Now that you have created the `site_meta` table, we want to join it with your cleaned air quality data.

1. What is the field or column can act as a unique key to join the two data frames? Save that column's name in your cleaned dataset in an object called `unique_key`.
2. Use a `left_join()` to combine your cleaned air quality dataframe with the `site_meta` dataframe. Save the result as a new dataframe called `complete_AQ` (i.e. a clear, descriptive name).
3. Create a new column called `pm25_adj` that multiplies the raw pm25 values by the station’s calibration factor.
4. Print the first 10 rows to check that the join worked correctly.

In [0]:
# Uncomment the line of code below and store your answer for 1.
# unique_key <- 
# YOUR CODE HERE
stop("No Answer Given!")

In [0]:
# Combine cleaned df with site_meta
# Create new column
# YOUR CODE HERE
stop("No Answer Given!")

In [0]:
# Print the first 10 rows
# YOUR CODE HERE
stop("No Answer Given!")

In [0]:
# your grade is based on the following tests: 

# Question 3 test 
if (unique_key!="location"){
    stop("You did not save the correct unique key.")
} else if (mean(complete_AQ$pm25_adj) %>% round(1) !=12.6 |
    var(complete_AQ$pm25_adj) %>% round(1) != 45.0){
    stop("You did not calculate the new column correctly.")
} else{
    print("Nice work, you joined the dataframes correctly.")
}
    


## Question 4: Pivots (4 points)

### Part A (3 points)
Now, we want to see how air quality changes over the course of a typical day at each station. To do this, we want to average the adjusted PM2.5 values by hour for each station, then pivot to a wide format so each location becomes its own column. Pivoting to wide format then lets us look at all four stations side by side, making it easier to compare their daily patterns directly.

1. Create an hour variable in your joined dataframe using `hour = lubridate::hour(timestamp)`. (Note: `hour()` extracts the hour of the day as an integer from 0 to 23)
2. Create a new dataframe, `hourly_pm25_avg`, that calculates the hourly average adjusted PM2.5 (`pm25_adj`) at each location. (Hint: group by hour and location, then summarise with `mean(pm25_adj, na.rm=TRUE)` to compute the averages)
3. Convert this hourly summary dataframe into wide format using `tidyr::pivot_wider()`. Name the new dataframe `hourly_pm25_wide` (i.e. a clear, descriptive name); it should have one row per hour and one column for each location, with the values filled in from the hourly averages you calculated in Step 2.
4. Print the first 10 rows of your wide format dataframe and confirm the following columns: `hour`, `East Station`, `North Station`, `South Station`, `West Station`.



In [0]:
# Create hour variable
# YOUR CODE HERE
stop("No Answer Given!")

In [0]:
# Create a new dataframe
# Uncomment the line of code below to get started
# hourly_pm25_avg <- 
# YOUR CODE HERE
stop("No Answer Given!")

In [0]:
# Convert to wide format
# Uncomment the line of code below to get started
# hourly_pm25_wide <- 
# YOUR CODE HERE
stop("No Answer Given!")

In [0]:
# Print the first 10 rows
# YOUR CODE HERE
# YOUR CODE HERE
stop("No Answer Given!")

In [0]:
# your grade is based on the following tests: 
# Question 4 test 
if (identical(unlist(hourly_pm25_wide[1,],use.names=F),c(0,NA,20.9,9.0,17.3))){
    stop("You did not calculate the new column and/or pivot the dataframe correctly.")
} else{
    print("Nice work, you joined the dataframes correctly.")
}
   

### Part B (1 point)

Provide 1-2 sentences describing any patterns you notice in PM2.5 concentrations during the first 10 hours of the day. This can include differences between stations or changes from midnight through the morning. **Keep your answer brief; it should not exceed 1 line if possible.**

YOUR ANSWER HERE

## Question 5: Plotting from Pivoted Data (4 points)

### Part A (3 points)

Now, we want to visualize these patterns using line charts. 

Using the pivoted dataframe from Question 4:

1. Convert the wide dataframe back into long format using `tidyr::pivot_longer()`. (See more information on `pivot_longer()` [here](https://cran.r-project.org/web/packages/tidyr/vignettes/pivot.html).) Call the new dataframe `hourly_pm25_long`. In the new dataframe, call the column holding the station names `location` and the column holding the values called `pm25_hourly`. This will take the four separate station columns and stack them into a single location column paired with their corresponding PM2.5 measurements.

2. Create a bar chart of the hourly mean adjusted PM2.5 at each location and overlay a smoothed trend line. Save the chart in an object called `my_plot`.
- Use `ggplot2::geom_col()` for the bars and add `ggplot2::geom_smooth()` for the trend line
- Create four subplots (one per location), using `ggplot2::facet_wrap(~location)`
- x-axis: hour (0–23)
- y-axis: hourly average PM2.5 (adjusted, µg/m³)
- Include clear axis labels and a descriptive title

In [0]:
# Convert wide dataframe back to long

# YOUR CODE HERE
stop("No Answer Given!")

# Create bar chart
# Uncomment the below line of code to start
# my_plot <- ggplot() 
# YOUR CODE HERE
stop("No Answer Given!")
# Uncomment the below line of code to print your chart once created
# my_plot


In [0]:
# your grade is based on the following tests: 
# Question 5 test 
if (nrow(hourly_pm25_long) != 96 |
    round(mean(hourly_pm25_long$pm25_hourly,na.rm=T),1) != 12.7 |
    round(var(hourly_pm25_long$pm25_hourly,na.rm=T),1) != 33.2){
    stop("You did not convert to a long dataframe correctly.")
} else if (my_plot$facet %>% inherits("FacetNull")){
    stop("You did not create subplots using facet_wrap.")
} else if (rlang::as_name(my_plot$mapping$x)!='hour' |
           rlang::as_name(my_plot$mapping$y)!='pm25_hourly'){
    stop("You did not use the correct x and y axis aesthetics.")
} else if (is.null(my_plot$labels$x) | is.null(my_plot$labels$y) | is.null(my_plot$labels$title)){
    stop("You did not set x and y axis labels and a chart title.")
} else{
    print("Nice work, you plotted from pivoted data correctly.")
}

### Part B (1 point)

Provide 1-2 sentences that describe any noticeable patterns, trends, or other key observations in the line charts. **Keep your answer brief; it should not exceed 1 line if possible.**

YOUR ANSWER HERE

---

## How to submit

When you've finished the problem set:

1. In the Colab menu bar, click **File → Save a copy in GitHub**.
2. In the dialog:
   - **Repository**: choose the repository GitHub Classroom created for you when you accepted this assignment (it will have your GitHub username in the name).
   - **Branch**: `main`.
   - **File path**: leave the filename as-is so it overwrites the starter notebook.
   - **Commit message**: something descriptive, e.g. *"Finished pset"*.
3. Click **OK**. Colab will push your completed notebook back to your assignment repository.

You can save to GitHub as often as you like — each save is just another commit. Your most recent commit before the due date is what gets graded.
